# Train YOLOv8 on Indian Traffic Data (Colab free T4)

Purpose: train a YOLOv8 detector on the India traffic dataset (classes:
`car, motorcycle, bus, truck, bicycle, auto`), evaluate per-class mAP50,
export ONNX + int8 artifacts, and package them into a model registry zip
ready to drop into the repo's `models/registry/`.

## Free T4 strategy
- **Session 1**: `yolov8n` @ 70 epochs, imgsz 640, batch 16. Fits comfortably
  in one free session (~60-80 epochs).
- **Session 2 (optional)**: `yolov8s` @ 50 epochs, batch 12. Run only if the
  n-model mAP is insufficient; it may not fit in one session.

## Runtime instructions
1. `Runtime` -> `Change runtime type` -> **T4 GPU**.
2. Put the dataset on Google Drive per the contract below.
3. Set `DRIVE_DATASET_ROOT` in cell 2, then `Run all` (skip the optional s-cell).

## Dataset contract
```
DRIVE_DATASET_ROOT/
  data.yaml          # train/val/test paths + names
  images/train|val|test/*.jpg
  labels/train|val|test/*.txt   # class_id cx cy w h (normalized)
  calibration/frames/*.jpg      # ~200 frames for int8 quantization
```

> NOTE: the class schema is PROVISIONAL. Adding classes later only requires
> appending ids at the END of `data.yaml` and retraining — no code changes here.


## Setup

Installs deps, mounts Drive (with a local fallback for structure testing),
and defines `DRIVE_DATASET_ROOT`. Edit the placeholder path before running.

In [ ]:
!pip install ultralytics onnx onnxruntime -q

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # EDIT: point at your dataset root on Google Drive
    DRIVE_DATASET_ROOT = '/content/drive/MyDrive/india_yolo_dataset'
except ImportError:
    print('Not in Colab: using local fallback path.')
    DRIVE_DATASET_ROOT = './india_yolo_dataset'

print('Dataset root:', DRIVE_DATASET_ROOT)

## Validate dataset

Checks images-vs-labels counts per split, asserts exactly 6 classes from
`data.yaml`, warns on empty label files and out-of-range boxes. Self-contained;
no repo imports (Colab has no checkout).

In [ ]:
import os, glob

CLASSES = ["car", "motorcycle", "bus", "truck", "bicycle", "auto"]

def validate(root):
    yaml_path = os.path.join(root, 'data.yaml')
    assert os.path.isfile(yaml_path), f'missing {yaml_path}'
    names = {}
    with open(yaml_path) as f:
        in_names = False
        for line in f:
            line = line.strip()
            if line.startswith('names:'):
                in_names = True; continue
            if in_names:
                if ':' in line:
                    k, v = line.split(':', 1)
                    names[int(k.strip())] = v.strip()
                elif not line:
                    break
    assert len(names) == 6, f'expected 6 classes, got {len(names)}'
    assert [names[i] for i in range(6)] == CLASSES,         f'class order must be {CLASSES}, got {[names.get(i) for i in range(6)]}'
    print('data.yaml OK: 6 classes in required order.')

    problems = 0
    for split in ('train', 'val', 'test'):
        imgs = sorted(glob.glob(os.path.join(root, 'images', split, '*.jpg')))
        lbls_dir = os.path.join(root, 'labels', split)
        lbls = {os.path.splitext(os.path.basename(p))[0]: p
                for p in glob.glob(os.path.join(lbls_dir, '*.txt'))}
        empty = 0
        for ip in imgs:
            stem = os.path.splitext(os.path.basename(ip))[0]
            lp = lbls.get(stem)
            if lp is None:
                print(f'WARN [{split}] no label file for {stem}'); problems += 1
                continue
            content = open(lp).read().strip()
            if not content:
                empty += 1
                continue
            for lineno, row in enumerate(content.splitlines(), 1):
                parts = row.split()
                cid = int(parts[0])
                if not (0 <= cid <= 5):
                    print(f'WARN [{split}] {stem}:{lineno} bad class id {cid}'); problems += 1
                vals = [float(x) for x in parts[1:]]
                if any(v < 0 or v > 1 for v in vals):
                    print(f'WARN [{split}] {stem}:{lineno} box outside [0,1]'); problems += 1
        if empty:
            print(f'WARN [{split}] {empty} empty label files')
        print(f'{split}: {len(imgs)} images, {len(lbls)} label files')
    calib = glob.glob(os.path.join(root, 'calibration', 'frames', '*.jpg'))
    print(f'calibration frames: {len(calib)} (want ~200)')
    if len(calib) == 0:
        print('WARN: calibration set missing; int8 quantization will fail later.')
        problems += 1
    print('validation done,', problems, 'problems')

validate(DRIVE_DATASET_ROOT)

## Train tier-low model: yolov8n @ 70 epochs

Fits one free T4 session (~60-80 epochs). Prints progress each epoch and a
`results.csv` snippet when done.

In [ ]:
from ultralytics import YOLO

model_n = YOLO('yolov8n.pt')
model_n.train(
    data=os.path.join(DRIVE_DATASET_ROOT, 'data.yaml'),
    epochs=70, imgsz=640, batch=16, patience=15,
    name='india_yolov8n',
)

run_dir_n = model_n.trainer.save_dir
results_csv = os.path.join(run_dir_n, 'results.csv')
print('\n--- results.csv (head/tail snippet) ---')
with open(results_csv) as f:
    lines = f.readlines()
for ln in lines[:2] + lines[-5:]:
    print(ln.rstrip())
BEST_N = os.path.join(run_dir_n, 'weights', 'best.pt')
print('best.pt:', BEST_N)

## Per-class mAP50 table

Runs validation and prints a formatted table. The `auto` row is highlighted
(`>>>`) because it is the class COCO lacks — watch it closely.

In [ ]:
metrics_n = YOLO(BEST_N).val(data=os.path.join(DRIVE_DATASET_ROOT, 'data.yaml'), verbose=False)

def print_map_table(metrics):
    # ultralytics Metric: .ap_class_index, .p, .r are arrays over present classes;
    # .ap50 is per-class AP@0.5; .map50 is overall scalar
    names = metrics.names  # {id: name}
    idx = metrics.box.ap_class_index
    header = f"{'class':<12} {'precision':>9} {'recall':>9} {'mAP50':>9}"
    print(header)
    print('-' * len(header))
    rows = {}
    for i, ci in enumerate(idx):
        name = names[int(ci)]
        p = float(metrics.box.p[i]); r = float(metrics.box.r[i])
        m = float(metrics.box.ap50[i])
        mark = '>>>' if name == 'auto' else '   '
        print(f"{mark} {name:<9} {p:>9.3f} {r:>9.3f} {m:>9.3f}")
        rows[name] = round(m, 4)
    print(f"\noverall mAP50: {float(metrics.box.map50):.3f}")
    return rows

per_class_n = print_map_table(metrics_n)
mAP50_n = round(float(metrics_n.box.map50), 4)

## OPTIONAL second session: yolov8s @ 50 epochs

Commented out by default so free-tier users don't blow the session budget.
Uncomment ONLY if yolov8n mAP is insufficient and you have a fresh session.

In [ ]:
# model_s = YOLO('yolov8s.pt')
# model_s.train(
#     data=os.path.join(DRIVE_DATASET_ROOT, 'data.yaml'),
#     epochs=50, imgsz=640, batch=12, patience=15,
#     name='india_yolov8s',
# )
# run_dir_s = model_s.trainer.save_dir
# BEST_S = os.path.join(run_dir_s, 'weights', 'best.pt')
# metrics_s = YOLO(BEST_S).val(data=os.path.join(DRIVE_DATASET_ROOT, 'data.yaml'), verbose=False)
# per_class_s = print_map_table(metrics_s)
# mAP50_s = round(float(metrics_s.box.map50), 4)

BEST_S = None  # set to best.pt path if you ran the s-model above

## Export ONNX (fp32)

Exports `best.pt` to ONNX (imgsz 640, opset 17, simplified) for every trained
size.

In [ ]:
sizes = {'n': BEST_N}
if BEST_S:
    sizes['s'] = BEST_S

onnx_paths = {}
for tag, pt in sizes.items():
    print(f'exporting yolov8{tag} ...')
    onnx_paths[tag] = YOLO(pt).export(format='onnx', imgsz=640, opset=17, simplify=True)
print(onnx_paths)

## int8 static quantization

Calibrates over `calibration/frames/*.jpg` (preprocess: resize 640x640,
BGR->RGB, /255, NCHW float32). Falls back cleanly to dynamic quantization
(static is preferred) if `quantize_static` fails in Colab.

In [ ]:
import cv2, numpy as np, traceback
from onnxruntime.quantization import CalibrationDataReader, quantize_static, quantize_dynamic, QuantType

class FrameReader(CalibrationDataReader):
    def __init__(self, frames, onnx_path):
        sess_inputs = ort_session_input_name(onnx_path)
        self.reiter = iter([self._pre(p) for p in frames])
        self.input_name = sess_inputs
    def _pre(self, path):
        img = cv2.imread(path)
        assert img is not None, f'cannot read {path}'
        img = cv2.resize(img, (640, 640))  # simple resize is fine for calibration
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        return {self.input_name: np.transpose(rgb, (2, 0, 1))[np.newaxis]}
    def get_next(self):
        return next(self.reiter, None)

def ort_session_input_name(onnx_path):
    import onnxruntime as ort
    return ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider']).get_inputs()[0].name

frames = sorted(glob.glob(os.path.join(DRIVE_DATASET_ROOT, 'calibration', 'frames', '*.jpg')))[:200]
assert frames, 'no calibration frames found'

int8_paths = {}
for tag, onx in onnx_paths.items():
    int8_out = onx.replace('.onnx', '-int8.onnx')
    try:
        reader = FrameReader(frames, onx)
        quantize_static(onx, int8_out, reader, weight_type=QuantType.QInt8)
        print(f'yolov8{tag}: STATIC int8 -> {int8_out}')
    except Exception:
        traceback.print_exc()
        print(f'yolov8{tag}: quantize_static failed, falling back to DYNAMIC '
              '(static is preferred; consider rerunning this cell)')
        quantize_dynamic(onx, int8_out, weight_type=QuantType.QInt8)
    int8_paths[tag] = int8_out
print(int8_paths)

## Package model registry

Creates `models/registry/india-yolov8{n,s}/` containing the ONNX models and
`metadata.json` matching the repo contract exactly.

In [ ]:
import datetime, json

registry_root = 'models/registry'
os.makedirs(registry_root, exist_ok=True)

def build_meta(name, quant, source_run, map50, per_class):
    return {
        'name': name,
        'classes': CLASSES,
        'imgsz': 640,
        'normalization': {'mean': [0, 0, 0], 'std': [255, 255, 255],
                          'layout': 'NCHW', 'color': 'RGB'},
        'quantization': quant,
        'source_run': source_run,
        'metrics': {'mAP50_overall': map50, 'per_class_mAP50': per_class},
        'provenance': {'dataset_root': DRIVE_DATASET_ROOT, 'epochs': EPOCHS[tag]},
    }

import shutil

EPOCHS = {'n': 70}
if BEST_S:
    EPOCHS['s'] = 50
source_run = datetime.date.today().isoformat()

# One metadata.json per size describes the deployed (int8) artifact;
# model.onnx stays alongside for fp32 inference.
for tag, onx in onnx_paths.items():
    d = os.path.join(registry_root, f'india-yolov8{tag}')
    os.makedirs(d, exist_ok=True)
    shutil.copy(onx, os.path.join(d, 'model.onnx'))
    shutil.copy(int8_paths[tag], os.path.join(d, 'model-int8.onnx'))
    map50 = mAP50_n if tag == 'n' else mAP50_s
    per_cls = per_class_n if tag == 'n' else per_class_s
    meta = build_meta(f'india-yolov8{tag}', 'int8', source_run, map50, per_cls)
    with open(os.path.join(d, 'metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)

for root_d, _, fs in os.walk(registry_root):
    for fn in fs:
        print(os.path.join(root_d, fn))

## Zip registry and copy back to Drive

Zips `models/registry/`, copies the archive to the Drive dataset root, and
prints download instructions.

In [ ]:
import shutil, zipfile

zip_base = 'india_model_registry'
shutil.make_archive(zip_base, 'zip', root_dir='.', base_dir='models/registry')
zip_path = zip_base + '.zip'
print('created:', zip_path)

drive_copy = None
try:
    dest = os.path.join(DRIVE_DATASET_ROOT, zip_path)
    shutil.copy(zip_path, dest)
    drive_copy = dest
    print('copied to Drive:', dest)
except Exception as e:
    print('Drive copy failed:', e)

print()
print('Download instructions:')
if drive_copy:
    print(f'  1. In Colab file browser or Drive, fetch: {drive_copy}')
else:
    print(f'  1. Download from the Colab file browser: {zip_path}')
print('  2. Unzip into the repo so that models/registry/india-yolov8*/ lands at repo root.')
print('  3. Follow the handoff checklist in the final cell.')

## Handoff checklist

1. Download the registry zip (see previous cell output).
2. Unzip its contents into the repo so the layout becomes:

   ```
   models/registry/
     india-yolov8n/{model.onnx, model-int8.onnx, metadata_fp32.json, metadata_int8.json}
     india-yolov8s/...   (if trained)
   ```
3. If your loader expects exactly `model.onnx` + `metadata.json` per directory,
   keep ONE variant per folder (rename `model-int8.onnx` -> `model.onnx` and
   `metadata_int8.json` -> `metadata.json`, moving the fp32 pair aside) — the
   runtime reads classes from `metadata.json`, never hardcoded.
4. Run the repo test suite (`make verify`, or the pytest equivalent on Windows).
5. Rerun `make eval` to confirm detection metrics against the new artifacts.
6. Update `docs/MASTER_PLAN.md` Phase T status board entry.

Remember: class schema is provisional. New vehicle types = append to end of
`data.yaml`, extend `core/domain.py::VehicleType`, retrain, ship new metadata.
